In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D 

DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

def construct_model(
  arch,
  train_embed_layer,
  act,
  width,
  hid_n_layers,
):
  # arch : "MLP", "ResNet", "DiffNet-Add", "DiffNet-Mul"
  # train_embed_layer : bool, whether to train the first layer
  # act : "tanh", "sin", "relu", "gelu"
  # width : int, number of hidden units in each layer
  # hid_n_layers : int, number of hidden layers
  if arch == "MLP":
    return MLP(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      residual=False
    )
  elif arch == "ResNet":
    return MLP(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      residual=True
    )
  elif arch == "DiffNet-Add":
    return DiffNet(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      scale_type='add'
    )
  elif arch == "DiffNet-Mul":
    return DiffNet(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      scale_type='mul'
    )
  elif arch.startswith("DiffNet-Const"):
    return DiffNet(
      act=act,
      train_embed_layer=train_embed_layer,
      width=width,
      hid_n_layers=hid_n_layers,
      scale_type='const',
      scale_const=float(arch.split('-')[-1]) if '-' in arch else 1.0
    )

class MLP(nn.Module):
  
  def __init__(self,
    act='relu',
    train_embed_layer=False,
    width=64,
    hid_n_layers=2,
    residual=False
  ):
    super(MLP, self).__init__()
    self.train_embed_layer = train_embed_layer
    self.width = width
    self.hid_n_layers = hid_n_layers
    self.residual = residual

    if act.lower() == 'relu':
      self.act = nn.ReLU()
    elif act.lower() == 'gelu':
      self.act = nn.GELU()
    elif act.lower() == 'tanh':
      self.act = nn.Tanh()
    elif act.lower() == 'sin':
      self.act = torch.sin

    self.embed_layer = nn.Linear(2, width, bias=True)
    if not self.train_embed_layer:
      self.embed_layer.weight.requires_grad = False
      self.embed_layer.bias.requires_grad = False
    
    self.hidden_layers = nn.ModuleList()
    for _ in range(hid_n_layers - 1):
      self.hidden_layers.append(nn.Linear(width, width, bias=True))
    
    self.output_layer = nn.Linear(width, 1, bias=True)
  
  def forward(self, scalar_input1, scalar_input2):
    scalar_input1 = scalar_input1.unsqueeze(-1)
    scalar_input2 = scalar_input2.unsqueeze(-1)

    combined_input = torch.cat((scalar_input1, scalar_input2), dim=-1)

    x = self.embed_layer(combined_input)
    x = torch.sin(x) 
    for layer in self.hidden_layers:
      h = layer(x)
      if self.residual:
        x = self.act(h) + x
      else:
        x = self.act(h)
    x = self.output_layer(x)
    return x  
class DiffNet(nn.Module):
  
  def __init__(self,
    act='relu',
    train_embed_layer=False,
    width=64,
    hid_n_layers=2,
    scale_type='add',
    scale_const=1.0,
  ):
    super(DiffNet, self).__init__()

    self.begin_embed_layer = nn.Linear(2, width, bias=True)
    if not train_embed_layer:
      self.begin_embed_layer.weight.requires_grad = False
      self.begin_embed_layer.bias.requires_grad = False
    
    self.embed_layers = nn.ModuleList()
    self.hidden_layers = nn.ModuleList()
    for i in range(hid_n_layers):
      embed_layer = nn.Linear(2, width, bias=True)
      if scale_type == 'add':
        embed_layer.weight.data *= (i + 2)
        embed_layer.bias.data *= (i + 2)
      elif scale_type == 'mul':
        embed_layer.weight.data *= 2**(i + 1)
        embed_layer.bias.data *= 2**(i + 1)
      elif scale_type == 'const' and scale_const != 1.0:
        embed_layer.weight.data *= scale_const
        embed_layer.bias.data *= scale_const
      
      if not train_embed_layer:
        embed_layer.weight.requires_grad = False
        embed_layer.bias.requires_grad = False
      self.embed_layers.append(embed_layer)

      self.hidden_layers.append(nn.Linear(2 * width, width, bias=True))
    
    self.output_layer = nn.Linear(width, 1, bias=True)

    if act.lower() == 'relu':
      self.act = nn.ReLU()
    elif act.lower() == 'gelu':
      self.act = nn.GELU()
    elif act.lower() == 'tanh':
      self.act = nn.Tanh()
    elif act.lower() == 'sin':
      self.act = torch.sin
  
  def forward(self, scalar_input1, scalar_input2):
    scalar_input1 = scalar_input1.unsqueeze(-1)
    scalar_input2 = scalar_input2.unsqueeze(-1)

    combined_input = torch.cat((scalar_input1, scalar_input2), dim=-1)

    x = self.begin_embed_layer(combined_input)
    x = torch.sin(x)
    for embed_layer, hidden_layer in zip(self.embed_layers, self.hidden_layers):
      h = embed_layer(combined_input)
      h = torch.sin(h)

      z = torch.cat((x, h), dim=-1)
      z = hidden_layer(z)
      x = self.act(z) + x

    x = self.output_layer(x)
    return x + 5


In [ ]:
def subset_product_mean(tensor, subset_size):
  coeffs = torch.zeros(subset_size + 1, device=tensor.device)
  coeffs[0] = 1.0

  size_vec = torch.arange(1, subset_size + 1, device=tensor.device)

  for i in range(len(tensor)):
    coeffs[1:] = (((i+1) - size_vec) / (i+1)) * coeffs[1:] + (size_vec / (i + 1)) * coeffs[:-1] * tensor[i]

  return coeffs[subset_size]

def compute_k_density(model, clique, exponent, num_samples, device):
  
  points = torch.rand([num_samples, clique], device=device)

  indices = torch.combinations(torch.arange(clique, device=device), r=2)

  pairs_first = points[:, indices[:, 0]]
  pairs_second = points[:, indices[:, 1]] 

  new_x = torch.minimum(pairs_first, pairs_second)
  new_y = torch.maximum(pairs_first, pairs_second)

  pair_densities = model(new_x, new_y).squeeze(-2) 

  pair_densities = torch.sigmoid(pair_densities)

  edge_densities = pair_densities.squeeze(2) 

  graph_densities = torch.prod(edge_densities, dim=1) 

  graph_power_densities = subset_product_mean(graph_densities, exponent) 

  return graph_power_densities

In [ ]:
import math

# Borrowed from https://github.com/katsura-jp/pytorch-cosine-annealing-with-warmup/blob/master/cosine_annealing_warmup/scheduler.py

class CosineAnnealingWarmupRestarts(torch.optim.lr_scheduler._LRScheduler):
  
  def __init__(self,
    optimizer: torch.optim.Optimizer,
    first_cycle_steps: int,
    cycle_mult: float = 1.,
    max_lr: float = 0.1,
    min_lr: float = 0.001,
    warmup_steps: int = 0,
    gamma: float = 1.,
    last_epoch: int = -1):

    assert warmup_steps < first_cycle_steps
     
    self.first_cycle_steps = first_cycle_steps # first cycle step size
    self.cycle_mult = cycle_mult # cycle steps magnification
    self.base_max_lr = max_lr # first max learning rate
    self.max_lr = max_lr # max learning rate in the current cycle
    self.min_lr = min_lr # min learning rate
    self.warmup_steps = warmup_steps # warmup step size
    self.gamma = gamma # decrease rate of max learning rate by cycle
        
    self.cur_cycle_steps = first_cycle_steps # first cycle step size
    self.cycle = 0 # cycle count
    self.step_in_cycle = last_epoch # step size of the current cycle
    
    super(CosineAnnealingWarmupRestarts, self).__init__(optimizer, last_epoch)
    
    self.init_lr()

  def init_lr(self):
    self.base_lrs = []
    for param_group in self.optimizer.param_groups:
      param_group['lr'] = self.min_lr
      self.base_lrs.append(self.min_lr)
  
  def get_lr(self):
    if self.step_in_cycle == -1:
      return self.base_lrs
    elif self.step_in_cycle < self.warmup_steps:
      return [(self.max_lr - base_lr) * self.step_in_cycle / self.warmup_steps + base_lr for base_lr in self.base_lrs]
    else:
      return [base_lr + \
              (self.max_lr - base_lr) \
              * (1 + math.cos(math.pi * (self.step_in_cycle-self.warmup_steps) \
                              / (self.cur_cycle_steps - self.warmup_steps))) / 2
              for base_lr in self.base_lrs]
  
  def step(self, epoch=None):
    if epoch is None:
      epoch = self.last_epoch + 1
      self.step_in_cycle = self.step_in_cycle + 1
      if self.step_in_cycle >= self.cur_cycle_steps:
        self.cycle += 1
        self.step_in_cycle = self.step_in_cycle - self.cur_cycle_steps
        self.cur_cycle_steps = int((self.cur_cycle_steps - self.warmup_steps) * self.cycle_mult) + self.warmup_steps
      else:
        if epoch >= self.first_cycle_steps:
          if self.cycle_mult == 1.:
            self.step_in_cycle = epoch % self.first_cycle_steps
            self.cycle = epoch // self.first_cycle_steps
          else:
            n = int(math.log((epoch / self.first_cycle_steps * (self.cycle_mult - 1) + 1), self.cycle_mult))
            self.cycle = n
            self.step_in_cycle = epoch - int(self.first_cycle_steps * (self.cycle_mult ** n - 1) / (self.cycle_mult - 1))
            self.cur_cycle_steps = self.first_cycle_steps * self.cycle_mult ** (n)
        else:
          self.cur_cycle_steps = self.first_cycle_steps
          self.step_in_cycle = epoch
                
      self.max_lr = self.base_max_lr * (self.gamma**self.cycle)
      self.last_epoch = math.floor(epoch)
      for param_group, lr in zip(self.optimizer.param_groups, self.get_lr()):
        param_group['lr'] = lr

In [ ]:
stat_quantile = torch.linspace(0, 1, 9, device=DEVICE, dtype=torch.float32)
def train_network(
    try_epochs=50000,
    total_epochs=500000,
    learning_rate=1e-4,
    mc_samples=128,
    test_mc_samples=10000,
    arch='DiffNet-Mul',
    act='gelu',
    network_width=64,
    network_depth=3,
    print_interval=10000,
  ):
  """
  Main training loop for the neural network.
  Args:
    p_target_value (float): The target value for the constraint E[f(x,y)] = p.
    num_epochs (int): Number of training epochs.
    learning_rate (float): Learning rate for the optimizer.
    mc_samples_objective (int): Number of MC samples for the objective term per epoch.
    mc_samples_constraint (int): Number of MC samples for the constraint term per epoch.
    lambda_penalty_value (float): Coefficient for the constraint penalty term.
    hidden_dim_core_net (int): Number of neurons in each hidden layer of CoreNet.
    num_hidden_layers_core_net (int): Number of hidden layers in CoreNet.
    print_interval (int): How often to print training progress.
  Returns:
    tuple: (trained_model, history_dict)
         - trained_model (CoreNet): The trained neural network.
         - history_dict (dict): A dictionary containing training metrics over epochs.
  """
  device = DEVICE

  # Initialize the core neural network
  core_model_instance = construct_model(
    arch=arch,
    train_embed_layer=True,
    act=act,
    width=network_width,
    hid_n_layers=network_depth
  ).to(device)

  optimizer = optim.Adam(core_model_instance.parameters(), lr=learning_rate)
  scheduler = CosineAnnealingWarmupRestarts(
    optimizer=optimizer,
    first_cycle_steps=10000,
    max_lr = learning_rate,
    min_lr = learning_rate / 100,
    warmup_steps=2000,
    gamma=1,
  )

  history = {
    'epoch': [],
    'k3_objective': [],
    'k4_objective': [],
    'objective_expectation': [],
    'learning_rate': []
  }

  print("Starting training with")
  print(f"Network: {arch}, Activation: {act}, Width: {network_width}, Depth: {network_depth}")
  print(f"Learning Rate: {learning_rate}, MC Samples: {mc_samples}")

  for epoch in range(try_epochs):
    core_model_instance.train() 
    optimizer.zero_grad() 

    k3_objective = compute_k_density(core_model_instance, 3, 2, mc_samples, device)
    k4_objective = compute_k_density(core_model_instance, 4, 1, mc_samples, device)
    
    total_loss = k3_objective - k4_objective

    total_loss.backward()
    optimizer.step()
    scheduler.step()

    history['epoch'].append(epoch)
    history['k3_objective'].append(k3_objective.item())
    history['k4_objective'].append(k4_objective.item())
    history['objective_expectation'].append(k3_objective.item() - k4_objective.item())
    history['learning_rate'].append(optimizer.param_groups[0]['lr'])

    if (epoch + 1) % print_interval == 0 or epoch == 0:
      print(f"Epoch {epoch + 1}/{total_epochs}, Objective Expectation: {k3_objective.item() - k4_objective.item():.6e} (P(K3): {k3_objective.item():.6e}, P(K4): {k4_objective.item():.6e})")

    del total_loss
    del k3_objective
    del k4_objective


  core_model_instance.eval() 
  with torch.no_grad():
    k3_objective = compute_k_density(core_model_instance, 3, 2, test_mc_samples, device)
    k4_objective = compute_k_density(core_model_instance, 4, 1, test_mc_samples, device)

  print(f"Initial Training Success: Test Objective Expectation {k3_objective.item() - k4_objective.item():.6e} (P(H): {k3_objective.item():.6e}, P(BP): {k4_objective.item():.6e})")
  
  del k3_objective
  del k4_objective

  for epoch in range(try_epochs, total_epochs):
    core_model_instance.train()
    optimizer.zero_grad() 

    k3_objective = compute_k_density(core_model_instance, 3, 2, mc_samples, device)
    k4_objective = compute_k_density(core_model_instance, 4, 1, mc_samples, device)
    
    total_loss = k3_objective - k4_objective

    total_loss.backward()
    optimizer.step()
    scheduler.step()

    # Logging training progress
    history['epoch'].append(epoch)
    history['k3_objective'].append(k3_objective.item())
    history['k4_objective'].append(k4_objective.item())
    history['objective_expectation'].append(k3_objective.item() - k4_objective.item())
    history['learning_rate'].append(optimizer.param_groups[0]['lr'])

    if (epoch + 1) % print_interval == 0 or epoch == 0:
      print(f"Epoch {epoch + 1}/{total_epochs}, Objective Expectation: {k3_objective.item() - k4_objective.item():.6e} (P(K3): {k3_objective.item():.6e}, P(K4): {k4_objective.item():.6e})")

  print("Training finished.")
  return core_model_instance, history

def ema(l, alpha=0.9):
  ema_values = [] 
  ema_values.append(l[0])

  for i in range(1, len(l)):
    ema_values.append((l[i] * alpha) + (ema_values[-1] * (1 - alpha)))
  return ema_values

def plot_training_history(history, k3_density, k4_density, diff):
  """Plots the training progress from the history dictionary."""
  if not history['epoch']:
    print("No history to plot. Ensure training ran for some epochs and print_interval was met.")
    return
  
  fig, axes = plt.subplots(2, 2, figsize=(12, 10))
  fig.suptitle(f"Training History - K3: {k3_density:4e}, K4: {k4_density:4e}, Diff: {diff:4e}", fontsize=16)

  axes[0, 0].plot(history['epoch'], history['objective_expectation'], label='Objective Expectation', color='blue', alpha=0.3)
  axes[0, 0].set_xlabel('Epoch')
  axes[0, 0].set_ylabel('Objective Expectation')

  axes[0, 1].plot(history['epoch'], history['learning_rate'], label='Learning Rate', color='orange')
  axes[0, 1].set_xlabel('Epoch')
  axes[0, 1].set_ylabel('Learning Rate')
  axes[0, 1].set_yscale('log')

  axes[1, 0].plot(history['epoch'], history['k3_objective'], label='K3 Density', color='green', alpha=0.3)
  axes[1, 0].set_xlabel('Epoch')
  axes[1, 0].set_ylabel('K3 Density')
  axes[1, 0].set_yscale('log')

  axes[1, 1].plot(history['epoch'], history['k4_objective'], label='K4 Density', color='red', alpha=0.3)
  axes[1, 1].set_xlabel('Epoch')
  axes[1, 1].set_ylabel('K4 Density')
  axes[1, 1].set_yscale('log')

  plt.tight_layout(rect=[0, 0, 1, 0.96])
  plt.show()

  return fig

def visualise_learnt_function_f_2d(model, device, k3_density, k4_density, resolution=50):
  """
  Visualises the learned symmetric function f(x,y) as a 2D image plot.

  Args:
    model (CoreNet): The trained core neural network.
    device (torch.device): The device the model is on.
    resolution (int): The number of points along x and y axes for the plot.
  """
  model.eval() 

  x_vals = np.linspace(0, 1, resolution)
  y_vals = np.linspace(0, 1, resolution)
  X, Y = np.meshgrid(x_vals, y_vals)

  grid_tensor_x = torch.tensor(X.flatten(), dtype=torch.float32, device=device)
  grid_tensor_y = torch.tensor(Y.flatten(), dtype=torch.float32, device=device)

  with torch.no_grad(): 
    Z_tensor = model(torch.minimum(grid_tensor_x, grid_tensor_y), torch.maximum(grid_tensor_x, grid_tensor_y))
    Z_tensor = torch.sigmoid(Z_tensor) 

  Z = Z_tensor.cpu().numpy().reshape(X.shape)

  fig, ax = plt.subplots(figsize=(8, 7)) 
  
  im = ax.imshow(Z, cmap='viridis', origin='lower', extent=[0, 1, 0, 1], vmin=0, vmax=1)

  plt.show()
  return fig

In [ ]:
from itertools import product
import yaml
import os
import gc

if __name__ == '__main__':
  
  NUM_EPOCHS = 50000 
  TRY_EPOCHS = 10000
  lr_list = [1e-3, 5e-4, 1e-4, 5e-5] 
  
  MC_SAMPLES = 2 ** 12
  TEST_MC_SAMPLES = 2 ** 15

  best_performance = 1.0

  hidden_dim_list = [256, 128, 64, 32] 
  hidden_n_layers_list = [5, 3, 1]

  arch_list = ['DiffNet-Mul', 'DiffNet-Add']
  train_embed_layer = True
  act_list = ['sin', 'gelu'] 
  PRINT_INTERVAL = 10000

  for hidden_dim_core_net, num_hidden_layers_core_net, act, arch, learning_rate in product(
    hidden_dim_list,
    hidden_n_layers_list,
    act_list,
    arch_list,
    lr_list
  ):
    print(f"\n--- Training with parameters: "
          f"arch={arch}, train_embed_layer={train_embed_layer}, act={act}, "
          f"hidden_dim_core_net={hidden_dim_core_net}, num_hidden_layers_core_net={num_hidden_layers_core_net}, "
          f"learning_rate={learning_rate}")
  
    trained_model, training_history = train_network(
      try_epochs=TRY_EPOCHS,
      total_epochs=NUM_EPOCHS,
      learning_rate=learning_rate,
      mc_samples=MC_SAMPLES,
      test_mc_samples=TEST_MC_SAMPLES,
      arch=arch,
      act=act,
      network_width=hidden_dim_core_net,
      network_depth=num_hidden_layers_core_net,
      print_interval=PRINT_INTERVAL,
    )

    if trained_model:
      current_device = next(trained_model.parameters()).device 
      trained_model.eval()

      print("\n--- Final Evaluation (using more samples) ---")
      with torch.no_grad(): 
        k3_objective_exp = compute_k_density(
          trained_model,
          3,
          2,
          TEST_MC_SAMPLES,
          current_device
        )
        k4_objective_exp = compute_k_density(
          trained_model,
          4,
          1,
          TEST_MC_SAMPLES,
          current_device
        )
        final_objective_exp = k3_objective_exp - k4_objective_exp

      best_performance = final_objective_exp.item()
      torch.save(trained_model.state_dict(), f"best_model_k3_k4_{arch}_{act}_{hidden_dim_core_net}_{num_hidden_layers_core_net}_{learning_rate}.pt")

      print(f"Post-training Objective E[K3]: {k3_objective_exp.item():.4e}")
      print(f"Post-training Objective E[K4]: {k4_objective_exp.item():.4e}")
      print(f"Post-training Objective E[K3] - E[K4]: {best_performance:.4e}")
      
      results_filename = f"results/k3_k4_{arch}_{act}_{hidden_dim_core_net}_{num_hidden_layers_core_net}_{learning_rate}.yaml"
      history_filename = f"results/history_k3_k4_{arch}_{act}_{hidden_dim_core_net}_{num_hidden_layers_core_net}_{learning_rate}.png"
      output_filename = f"results/output_k3_k4_{arch}_{act}_{hidden_dim_core_net}_{num_hidden_layers_core_net}_{learning_rate}.png"
      results_data = {
        'arch': arch,
        'train_embed_layer': train_embed_layer,
        'act': act,
        'hidden_dim_core_net': hidden_dim_core_net,
        'num_hidden_layers_core_net': num_hidden_layers_core_net,
        'k3_objective_exp': k3_objective_exp.item(),
        'k4_objective_exp': k4_objective_exp.item(),
        'final_objective_exp': final_objective_exp.item(),
      }
      with open(results_filename, 'w') as f:
        yaml.dump(results_data, f)        
        
      results_csv_filename = "k3_k4_res.csv"
      if not os.path.exists(results_csv_filename):
        with open(results_csv_filename, 'w') as f:
          f.write("arch,train_embed_layer,act,hidden_dim_core_net,num_hidden_layers_core_net,"
                  "constraint_exp,objective_exp,final_output_std\n")
        with open(results_csv_filename, 'a') as f:
          f.write(f"{arch},{train_embed_layer},{act},"
                  f"{hidden_dim_core_net},{num_hidden_layers_core_net},"
                  f"{final_objective_exp.item():.4f},")
      
      fig = plot_training_history(training_history, k3_objective_exp.item(), k4_objective_exp.item(), best_performance)
      fig.savefig(history_filename, bbox_inches='tight')
      plt.close(fig)

      print("\n--- Visualizing the learned function f(x,y) ---")
      fig = visualise_learnt_function_f_2d(trained_model, current_device, k3_objective_exp.item(), k4_objective_exp.item(), resolution=2 ** 8)
      fig.savefig(output_filename, bbox_inches='tight')
      plt.close(fig)
      
      gc.collect()